In [69]:
import json, os, sys
sys.path.append("../")
import numpy as np
from tqdm import tqdm
from steps.utils.generic import read_json_or_jsonl, write_to_json, maybe_create_folder
from steps.utils.kg_based_qg import KGBasedQGUtils, KGBasedQGChecker

In [84]:
DATE = "2Dec2025"

decontextualized_facts_folder = f"/scratch/lamdo/IRB/runs/{DATE}/step2_2"
fact_groundedness_folder = f"/scratch/lamdo/IRB/runs/{DATE}/step3"
extracted_kg_folder = f"/scratch/lamdo/IRB/runs/{DATE}/step4_extracted_kg"


In [88]:
num_keypoints = []
num_kgs = []

kg_based_qg_checker = KGBasedQGChecker(
        minicheck_model_name = None,
        minicheck_cache_dir = None
    )

files = os.listdir(decontextualized_facts_folder)
files = [file for file in files if file.endswith('.json')]

decontextualized_facts_files_full_path = [os.path.join(decontextualized_facts_folder, file) for file in files]
fact_groundedness_files_full_path = [os.path.join(fact_groundedness_folder, file) for file in files]
extracted_kg_files_full_path = [os.path.join(extracted_kg_folder, file) for file in files]


for dff_file_path, fgf_file_path, ekg_file_path in tqdm(zip(decontextualized_facts_files_full_path, 
                                                    fact_groundedness_files_full_path,
                                                    extracted_kg_files_full_path), total = len(files)):
    try:
        dff_data = read_json_or_jsonl(dff_file_path)
        fgf_data = read_json_or_jsonl(fgf_file_path)
        ekg_data = read_json_or_jsonl(ekg_file_path)
    except FileNotFoundError:
        continue

    assert dff_data.get("title") == fgf_data.get("title") == ekg_data.get("title")

    keypoints_mapper = dff_data.get("keypoints_mapper")
    groundedness_check = fgf_data.get("groundedness_check")


    if not keypoints_mapper or not groundedness_check: continue

    groundedness_check = {tuple(k.split("--__--")): v for k,v in groundedness_check.items()}
    good_keypoints = set([])
    for k, v in groundedness_check.items():
        if v:
            good_keypoints.add(f"{k[0]}--__--{k[-1]}")

    keypoints_mapper = {int(k): v for k,v in keypoints_mapper.items()}
    keypoints_mapper_filtered = {}

    for fact_id, keypoints in keypoints_mapper.items():
        to_update = []
        for kp_index, kp in enumerate(keypoints):
            if f"{fact_id}--__--{kp_index}" in good_keypoints: to_update.append(kp)
        
        if to_update:
            keypoints_mapper_filtered[fact_id] = to_update
    
    keypoints_mapper = keypoints_mapper_filtered
    if not keypoints_mapper: continue

    num_keypoints.append(len(keypoints_mapper))

    good_kg_count = 0
    for fact_id, keypoints in keypoints_mapper.items():
        good_kg = False

        if str(fact_id) not in ekg_data["fact_kg_mapper"]: continue
        extracted_kg = ekg_data["fact_kg_mapper"][str(fact_id)]
        graph_completenesss = kg_based_qg_checker.check_completeness_of_extracted_kg(
            knowledge_graph = extracted_kg, 
            keypoints = keypoints, 
            word_check_threshold = 0.1)
        
        if graph_completenesss: 
            good_kg = True
            
        if good_kg: good_kg_count += 1
    num_kgs.append(good_kg_count)

100%|██████████| 1642/1642 [00:02<00:00, 635.01it/s]


In [89]:
np.mean(num_kgs) / np.mean(num_keypoints)

np.float64(0.9995966115369099)

In [80]:
np.sum(num_keypoints)

np.int64(441)

In [81]:
num_kgs[:10]

[1, 1, 1, 4, 4, 2, 0, 0, 2, 1]

In [66]:
dff_data

{'title': '12 Gifts of Christmas',
 'wiki_url': 'https://en.wikipedia.org/?curid=75701597',
 'topics': ['Culture.Media.Media*',
  'Culture.Philosophy_and_religion',
  'Culture.Media.Films'],
 'create_timestamp': '2024-01-02T09:15:00Z',
 'timestamp': '2025-08-27T04:24:47Z',
 'keypoints_mapper': {'0': ['12 Gifts of Christmas is a 2015 American Christmas romantic comedy television film that was directed by Peter Sullivan and written by Peter Sullivan, Lynn Grant Beck, and Jennifer Notas Shapiro.'],
  '2': ["The 2015 American Christmas romantic comedy television film '12 Gifts of Christmas,' directed by Peter Sullivan and starring Katrina Law, Aaron O'Connell, and Donna Mills, premiered on Hallmark Channel on November 26, 2015, and later re-aired every Christmas season during the Countdown to Christmas programming block."]}}

In [67]:
fgf_data

{'title': '12 Gifts of Christmas',
 'wiki_url': 'https://en.wikipedia.org/?curid=75701597',
 'topics': ['Culture.Media.Media*',
  'Culture.Philosophy_and_religion',
  'Culture.Media.Films'],
 'create_timestamp': '2024-01-02T09:15:00Z',
 'timestamp': '2025-08-27T04:24:47Z',
 'groundedness_check': {'0--__--https://www.rottentomatoes.com/m/12_gifts_of_christmas_2015--__--0': True,
  '2--__--https://www.hallmarkchannel.com/12-gifts-of-christmas--__--0': False}}

In [68]:
ekg_data

{'title': '12 Gifts of Christmas',
 'wiki_url': 'https://en.wikipedia.org/?curid=75701597',
 'topics': ['Culture.Media.Media*',
  'Culture.Philosophy_and_religion',
  'Culture.Media.Films'],
 'create_timestamp': '2024-01-02T09:15:00Z',
 'timestamp': '2025-08-27T04:24:47Z',
 'fact_kg_mapper': {'0': [{'head': '12 Gifts of Christmas',
    'head_type': 'Film',
    'relation': 'is a',
    'tail': '2015 American Christmas romantic comedy television film',
    'tail_type': 'Film genre and year',
    'head_coverage': [0],
    'tail_coverage': [0]},
   {'head': '12 Gifts of Christmas',
    'head_type': 'Film',
    'relation': 'was directed by',
    'tail': 'Peter Sullivan',
    'tail_type': 'Person',
    'head_coverage': [0],
    'tail_coverage': [0]},
   {'head': '12 Gifts of Christmas',
    'head_type': 'Film',
    'relation': 'was written by',
    'tail': 'Peter Sullivan',
    'tail_type': 'Person',
    'head_coverage': [0],
    'tail_coverage': [0]},
   {'head': '12 Gifts of Christmas',
   

In [40]:
keypoints_mapper.keys()

dict_keys([3])

In [41]:
ekg_data["fact_kg_mapper"]

{'3': [{'head': 'Agueda Sena',
   'head_type': 'Person',
   'relation': 'was born on',
   'tail': '16 June 1927',
   'tail_type': 'Date',
   'head_coverage': [0],
   'tail_coverage': [0]},
  {'head': 'Agueda Sena',
   'head_type': 'Person',
   'relation': 'was born in',
   'tail': 'Lisbon',
   'tail_type': 'Place',
   'head_coverage': [0],
   'tail_coverage': [0]},
  {'head': 'Agueda Sena',
   'head_type': 'Person',
   'relation': 'was daughter of',
   'tail': 'Nazaria Celsa Camacho Quiroga de Vasconcelos',
   'tail_type': 'Person',
   'head_coverage': [0],
   'tail_coverage': [0]},
  {'head': 'Nazaria Celsa Camacho Quiroga de Vasconcelos',
   'head_type': 'Person',
   'relation': 'is also known as',
   'tail': 'Celsa Camacho',
   'tail_type': 'Person',
   'head_coverage': [0],
   'tail_coverage': [0]},
  {'head': 'Nazaria Celsa Camacho Quiroga de Vasconcelos',
   'head_type': 'Person',
   'relation': 'was Bolivian mother of',
   'tail': 'Agueda Sena',
   'tail_type': 'Person',
   'hea